# Konux Case Study – Full Analysis with Visualizations

This notebook fully solves the Konux case study:
- Preliminary signal analysis
- Train passage segmentation
- Feature extraction
- Train-type clustering
- Speed proxy estimation
- **All visualizations included**

Sampling rate: **2 kHz**

## 1. Imports and Global Parameters

In [1]:
import os, glob
import numpy as np
import pandas as pd
import scipy.signal as sig
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

FS = 2000.0

## 2. Load Binary Acceleration Data

In [ ]:
def read_trace(path):
    return np.fromfile(path, dtype=np.float32)

DATA_DIR = '/home/mamunds/job/pyspark/konux/data'  # change if needed
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.bin')))
print(f'Found {len(files)} files')

## 3. Signal Preprocessing

In [ ]:
def preprocess(x, fs=FS):
    x = x - np.median(x)
    sos = sig.butter(4, [5, 400], btype='bandpass', fs=fs, output='sos')
    return sig.sosfiltfilt(sos, x)

## 4. Train Passage Segmentation

In [ ]:
def segment_passage(x, fs=FS, win_s=0.1):
    win = int(win_s * fs)
    rms = np.sqrt(sig.convolve(x**2, np.ones(win)/win, mode='same'))
    med = np.median(rms)
    mad = np.median(np.abs(rms - med)) + 1e-12
    thr = med + 6 * mad
    idx = np.where(rms > thr)[0]
    if len(idx) == 0:
        return x, rms, thr
    splits = np.where(np.diff(idx) > 1)[0]
    starts = np.r_[idx[0], idx[splits + 1]]
    ends = np.r_[idx[splits], idx[-1]]
    k = np.argmax(ends - starts)
    s, e = starts[k], ends[k]
    pad = int(0.5 * fs)
    return x[max(0,s-pad):min(len(x), e+pad)], rms, thr

## 5. Visualization – Raw vs Filtered Signal

In [ ]:
raw = read_trace(files[0])
filt = preprocess(raw)
t = np.arange(len(raw)) / FS

plt.figure(figsize=(12,4))
plt.plot(t, raw, alpha=0.5, label='Raw')
plt.plot(t, filt, label='Filtered')
plt.xlabel('Time [s]'); plt.ylabel('Acceleration [g]')
plt.title('Raw vs Filtered Signal')
plt.legend(); plt.show()

## 6. Visualization – Train Passage Detection

In [ ]:
seg, rms, thr = segment_passage(filt)
t = np.arange(len(filt)) / FS

plt.figure(figsize=(12,4))
plt.plot(t, rms, label='RMS envelope')
plt.axhline(thr, color='r', linestyle='--', label='Threshold')
plt.xlabel('Time [s]'); plt.ylabel('RMS [g]')
plt.title('Train Passage Detection')
plt.legend(); plt.show()

## 7. Feature Extraction

In [ ]:
def extract_features(x, fs=FS):
    f, Pxx = sig.welch(x, fs=fs, nperseg=min(4096, len(x)))
    env = np.abs(sig.hilbert(x))
    fe, Pe = sig.welch(env, fs=fs, nperseg=min(4096, len(env)))
    mask = (fe >= 1) & (fe <= 30)
    dom_env_freq = fe[mask][np.argmax(Pe[mask])]
    return {
        'duration_s': len(x)/fs,
        'rms': np.sqrt(np.mean(x**2)),
        'peak': np.max(np.abs(x)),
        'spec_centroid': np.sum(f*Pxx)/np.sum(Pxx),
        'env_dom_freq': dom_env_freq
    }

## 8. Process All Files

In [ ]:
rows = []
for fpath in files:
    raw = read_trace(fpath)
    x = preprocess(raw)
    seg, _, _ = segment_passage(x)
    feats = extract_features(seg)
    feats['file'] = os.path.basename(fpath)
    rows.append(feats)

df = pd.DataFrame(rows)
df

## 9. Visualization – Train Type Clustering

In [ ]:
X = df.drop(columns=['file']).values
Xs = StandardScaler().fit_transform(X)
Z = PCA(n_components=2).fit_transform(Xs)

kmeans = KMeans(n_clusters=min(4, len(df)), random_state=0, n_init='auto')
df['cluster'] = kmeans.fit_predict(Z)

plt.figure(figsize=(6,6))
plt.scatter(Z[:,0], Z[:,1], c=df['cluster'])
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('PCA – Train Type Clusters')
plt.show()

## 10. Visualization – Speed Proxy Comparison

In [ ]:
df_sorted = df.sort_values('env_dom_freq', ascending=False)
plt.figure(figsize=(8,4))
plt.barh(df_sorted['file'], df_sorted['env_dom_freq'])
plt.xlabel('Dominant Envelope Frequency [Hz]')
plt.title('Relative Train Speed Proxy')
plt.gca().invert_yaxis()
plt.show()

## 11. Save Results

In [ ]:
df.to_csv('train_analysis_results.csv', index=False)
print('Results saved to train_analysis_results.csv')